# 02a - Dataset Creation

**Thesis:** Structured Subsystem-Aware Feature Representations for Predictive Maintenance in Metro Systems
**Author:** Emirhan Kurtulus, TU Wien

This notebook turns the raw MetroAT `train/` and `test/` parquet files into the two model-ready representations of the MetroAT framework (Steiner et al., RAMS 2026; Helm's code in `references/stefan/`):
1. **Windowed feature dataset** (RF input): 6-min windows (Wfeat = 350 s) with summary features per sensor and the 6.2 h early-warning label (Wlabel = 22326 s)
2. **Sequence dataset** (LSTM-AE input): 23-min sequences (Wseq = 1402 s), z-score normalised with train statistics

The official MetroAT train/test split is kept: train windows come only from `train/`, test windows only from `test/`.

**Outputs:** `outputs/supervised/train.parquet`, `test.parquet` and `outputs/unsupervised/` (`train.npy`, `val.npy`, `test*.npy`, `metadata.json`). The baselines trained on these datasets are in NB02b.

In [1]:
import sys, os, gc, bisect, itertools, json, warnings
import numpy as np
import pandas as pd
import glob
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

ROOT = os.path.abspath('..')
TRAIN_GLOB = os.path.join(ROOT, 'train', '**', '*.parquet')
TEST_GLOB  = os.path.join(ROOT, 'test',  '**', '*.parquet')
OUT_DIR    = os.path.join(ROOT, 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

RNG = np.random.RandomState(42)

# ── Steiner's best config ──
WINDOW_S   = 350      # 6 min feature aggregation window
HORIZON_S  = 22326    # 6.2 h failure label shift
TS_COL     = 'TIMESTAMP'
FAIL_COL   = 'TRAIN_IS_IN_FAILURE'
MAINT_COL  = 'TRAIN_IS_IN_MAINTENANCE'
FAIL_TYPE  = 'TRAIN_FAILURE_TYPE'

# ── Sensor classification (matching Steiner's 95 sensors) ──
# 74 analog sensors: continuous values -> min/max/mean/sum
ANALOG_SENSORS = [
    # APU (2)
    'CW1_MAIN_RESERVOIR_PRESSURE', 'CW2_MAIN_RESERVOIR_PRESSURE',
    # Brake cylinder pressure (12)
    'CW1_BRAKE_CYLINDER_PRESSURE_BOGIE1', 'CW1_BRAKE_CYLINDER_PRESSURE_BOGIE2',
    'CW2_BRAKE_CYLINDER_PRESSURE_BOGIE1', 'CW2_BRAKE_CYLINDER_PRESSURE_BOGIE2',
    'MW1_BRAKE_CYLINDER_PRESSURE_BOGIE1', 'MW1_BRAKE_CYLINDER_PRESSURE_BOGIE2',
    'MW2_BRAKE_CYLINDER_PRESSURE_BOGIE1', 'MW2_BRAKE_CYLINDER_PRESSURE_BOGIE2',
    'MW3_BRAKE_CYLINDER_PRESSURE_BOGIE1', 'MW3_BRAKE_CYLINDER_PRESSURE_BOGIE2',
    'MW4_BRAKE_CYLINDER_PRESSURE_BOGIE1', 'MW4_BRAKE_CYLINDER_PRESSURE_BOGIE2',
    # Spring brake pressure (12)
    'CW1_SPRING_BRAKE_PRESSURE_BOGIE1', 'CW1_SPRING_BRAKE_PRESSURE_BOGIE2',
    'CW2_SPRING_BRAKE_PRESSURE_BOGIE1', 'CW2_SPRING_BRAKE_PRESSURE_BOGIE2',
    'MW1_SPRING_BRAKE_PRESSURE_BOGIE1', 'MW1_SPRING_BRAKE_PRESSURE_BOGIE2',
    'MW2_SPRING_BRAKE_PRESSURE_BOGIE1', 'MW2_SPRING_BRAKE_PRESSURE_BOGIE2',
    'MW3_SPRING_BRAKE_PRESSURE_BOGIE1', 'MW3_SPRING_BRAKE_PRESSURE_BOGIE2',
    'MW4_SPRING_BRAKE_PRESSURE_BOGIE1', 'MW4_SPRING_BRAKE_PRESSURE_BOGIE2',
    # Proportional valve pressure (12)
    'CW1_PROPORTIONAL_VALVE_PRESSURE_BOGIE1', 'CW1_PROPORTIONAL_VALVE_PRESSURE_BOGIE2',
    'CW2_PROPORTIONAL_VALVE_PRESSURE_BOGIE1', 'CW2_PROPORTIONAL_VALVE_PRESSURE_BOGIE2',
    'MW1_PROPORTIONAL_VALVE_PRESSURE_BOGIE1', 'MW1_PROPORTIONAL_VALVE_PRESSURE_BOGIE2',
    'MW2_PROPORTIONAL_VALVE_PRESSURE_BOGIE1', 'MW2_PROPORTIONAL_VALVE_PRESSURE_BOGIE2',
    'MW3_PROPORTIONAL_VALVE_PRESSURE_BOGIE1', 'MW3_PROPORTIONAL_VALVE_PRESSURE_BOGIE2',
    'MW4_PROPORTIONAL_VALVE_PRESSURE_BOGIE1', 'MW4_PROPORTIONAL_VALVE_PRESSURE_BOGIE2',
    # Pneumatic braking force (12)
    'CW1_PNEUMATIC_BRAKING_FORCE_BOGIE1', 'CW1_PNEUMATIC_BRAKING_FORCE_BOGIE2',
    'CW2_PNEUMATIC_BRAKING_FORCE_BOGIE1', 'CW2_PNEUMATIC_BRAKING_FORCE_BOGIE2',
    'MW1_PNEUMATIC_BRAKING_FORCE_BOGIE1', 'MW1_PNEUMATIC_BRAKING_FORCE_BOGIE2',
    'MW2_PNEUMATIC_BRAKING_FORCE_BOGIE1', 'MW2_PNEUMATIC_BRAKING_FORCE_BOGIE2',
    'MW3_PNEUMATIC_BRAKING_FORCE_BOGIE1', 'MW3_PNEUMATIC_BRAKING_FORCE_BOGIE2',
    'MW4_PNEUMATIC_BRAKING_FORCE_BOGIE1', 'MW4_PNEUMATIC_BRAKING_FORCE_BOGIE2',
    # Load pressure (12)
    'CW1_LOAD_PRESSURE_BOGIE1', 'CW1_LOAD_PRESSURE_BOGIE2',
    'CW2_LOAD_PRESSURE_BOGIE1', 'CW2_LOAD_PRESSURE_BOGIE2',
    'MW1_LOAD_PRESSURE_BOGIE1', 'MW1_LOAD_PRESSURE_BOGIE2',
    'MW2_LOAD_PRESSURE_BOGIE1', 'MW2_LOAD_PRESSURE_BOGIE2',
    'MW3_LOAD_PRESSURE_BOGIE1', 'MW3_LOAD_PRESSURE_BOGIE2',
    'MW4_LOAD_PRESSURE_BOGIE1', 'MW4_LOAD_PRESSURE_BOGIE2',
    # Load signal (6)
    'CW1_LOAD_SIGNAL', 'CW2_LOAD_SIGNAL',
    'MW1_LOAD_SIGNAL', 'MW2_LOAD_SIGNAL', 'MW3_LOAD_SIGNAL', 'MW4_LOAD_SIGNAL',
    # Traction (4)
    'MW1_ENERGY_BRAKING_RESISTANCE', 'MW2_ENERGY_BRAKING_RESISTANCE',
    'MW3_ENERGY_BRAKING_RESISTANCE', 'MW4_ENERGY_BRAKING_RESISTANCE',
    # Context (2)
    'TRAIN_SPEED_ACTUAL', 'AMBIENT_TEMPERATURE',
]

# 21 binary sensors: 0/1 flags -> active_s/flips
BINARY_SENSORS = [
    # Pneumatic brake active (2)
    'CW1_PNEUMATIC_BRAKE_ACTIVE', 'CW2_PNEUMATIC_BRAKE_ACTIVE',
    # Spring brake active (4)
    'CW1_SPRING_BRAKE_ACTIVE_BOGIE1', 'CW1_SPRING_BRAKE_ACTIVE_BOGIE2',
    'CW2_SPRING_BRAKE_ACTIVE_BOGIE1', 'CW2_SPRING_BRAKE_ACTIVE_BOGIE2',
    # Proportional valve pressure available (12)
    'CW1_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'CW1_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'CW2_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'CW2_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'MW1_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'MW1_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'MW2_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'MW2_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'MW3_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'MW3_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'MW4_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'MW4_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    # Compressor running (2)
    'CW1_COMPRESSOR_RUNNING', 'CW2_COMPRESSOR_RUNNING',
    # Brake signal (1)
    'TRAIN_BRAKE_SIGNAL',
]

ALL_SENSORS = ANALOG_SENSORS + BINARY_SENSORS

print(f'Analog sensors: {len(ANALOG_SENSORS)}')
print(f'Binary sensors: {len(BINARY_SENSORS)}')
print(f'Total sensors:  {len(ALL_SENSORS)}')
print(f'Expected features: {len(ANALOG_SENSORS)*4} analog + {len(BINARY_SENSORS)*2} binary + 2 asset = {len(ANALOG_SENSORS)*4 + len(BINARY_SENSORS)*2 + 2}')
print(f'Window: {WINDOW_S}s ({WINDOW_S/60:.1f} min) | Horizon: {HORIZON_S}s ({HORIZON_S/3600:.1f} h)')

# ── Cache flags: expensive steps reuse saved outputs unless explicitly rebuilt ──
# Loading all raw sensor data needs ~7 GB of RAM; labels only need the cached features.
REBUILD_FEATURES = False   # re-create windowed features from raw parquet
REBUILD_SEQUENCES = False  # re-create LSTM-AE sequences from raw parquet

Analog sensors: 74
Binary sensors: 21
Total sensors:  95
Expected features: 296 analog + 42 binary + 2 asset = 340
Window: 350s (5.8 min) | Horizon: 22326s (6.2 h)


## 1. Supervised Feature Dataset Creation

Replicates Steiner/Helm's pipeline (`references/stefan/create_supervised_feature_datasets.py`):
- Epoch-aligned non-overlapping windows (350 s)
- Filter out maintenance periods
- Analog: min/max/mean/sum per window
- Binary: active_s (seconds active) / flips (state transitions) per window
- Asset lookback: days_since_last_failure, days_since_last_revision
- **Shifted label:** label = 1 if a failure overlaps [window_end, window_end + 6.2 h) (Helm's `create_shifted_labels`: max of the failure flag over that interval)
- **Filter rule:** keep label = 1 or (label = 0 and not in failure). Because in-failure windows are already positive, this removes nothing, exactly as in Helm's code. (Helm's thesis text, p87, describes dropping ambiguous in-failure windows, but the code does not.)

Windowed features are cached in `outputs/supervised/features_*_unlabeled.parquet`, so labels can be recomputed without reloading the raw data (set `REBUILD_FEATURES = True` to rebuild).

In [2]:
# ── Helper functions ──

def get_failure_events(df):
    """Extract continuous failure event blocks."""
    mask = df[FAIL_COL].astype(bool)
    grp = (mask != mask.shift()).cumsum()
    events = (
        df[mask].groupby(grp[mask])
        .agg(start=(TS_COL, 'min'), end=(TS_COL, 'max'),
             event_type=(FAIL_TYPE, 'first'), n_rows=(TS_COL, 'count'))
        .reset_index(drop=True)
    )
    return events


def get_maintenance_events(df):
    """Extract continuous maintenance event blocks."""
    mask = df[MAINT_COL].astype(bool)
    grp = (mask != mask.shift()).cumsum()
    events = (
        df[mask].groupby(grp[mask])
        .agg(start=(TS_COL, 'min'), end=(TS_COL, 'max'), n_rows=(TS_COL, 'count'))
        .reset_index(drop=True)
    )
    return events


def create_windowed_features(df, window_s, analog_sensors, binary_sensors):
    """Create epoch-aligned non-overlapping windowed features using vectorized ops."""
    df = df.set_index(TS_COL).sort_index()
    
    # Pre-compute flip indicators for binary sensors
    flip_cols = []
    for col in binary_sensors:
        fcol = f'_flip_{col}'
        df[fcol] = df[col].diff().abs().fillna(0)
        flip_cols.append(fcol)
    
    # Epoch-aligned resampling
    resampler = df.resample(f'{window_s}s', origin='epoch')
    
    # Analog aggregations
    print('  Computing analog features...')
    analog_min  = resampler[analog_sensors].min()
    analog_max  = resampler[analog_sensors].max()
    analog_mean = resampler[analog_sensors].mean()
    analog_sum  = resampler[analog_sensors].sum()
    
    analog_min.columns  = [f'{c}_min' for c in analog_sensors]
    analog_max.columns  = [f'{c}_max' for c in analog_sensors]
    analog_mean.columns = [f'{c}_mean' for c in analog_sensors]
    analog_sum.columns  = [f'{c}_sum' for c in analog_sensors]
    
    # Binary aggregations
    print('  Computing binary features...')
    binary_active = resampler[binary_sensors].sum()
    binary_flips  = resampler[flip_cols].sum()
    
    binary_active.columns = [f'{c}_active_s' for c in binary_sensors]
    binary_flips.columns  = [f'{c}_flips' for c in binary_sensors]
    
    # Count rows per window (to filter empty windows)
    counts = resampler[analog_sensors[0]].count().rename('_n_rows')
    
    # Combine
    feats = pd.concat([analog_min, analog_max, analog_mean, analog_sum,
                       binary_active, binary_flips, counts], axis=1)
    
    # Remove empty windows
    feats = feats[feats['_n_rows'] > 0].drop(columns=['_n_rows'])
    
    # Add timestamp and window_end columns
    feats['window_end'] = feats.index + pd.Timedelta(seconds=window_s)
    feats = feats.reset_index().rename(columns={TS_COL: 'timestamp'})
    
    print(f'  Created {len(feats)} windows with {len(feats.columns)-2} features')
    return feats


def compute_shifted_labels(window_ends, failure_events, horizon_s):
    """Helm/Steiner label (references/stefan/create_supervised_feature_datasets.py, create_shifted_labels):
    label = max(TRAIN_IS_IN_FAILURE) over [window_end, window_end + horizon), i.e. 1 if any failure overlaps that interval.
    Windows inside a failure are therefore positive, so the filter rule below removes nothing (same as in Helm's code)."""
    we_arr = pd.to_datetime(window_ends).values
    labels = np.zeros(len(we_arr), dtype=np.int8)
    horizon_td = np.timedelta64(horizon_s, 's')
    for _, fe in failure_events.iterrows():
        mask = (np.datetime64(fe['start']) < we_arr + horizon_td) & (np.datetime64(fe['end']) >= we_arr)
        labels[mask] = 1
    return labels


def compute_current_failure(window_ends, failure_events):
    """For each window_end, in_failure=1 if it falls within any failure interval."""
    we_arr = pd.to_datetime(window_ends).values
    labels = np.zeros(len(we_arr), dtype=np.int8)
    
    for _, fe in failure_events.iterrows():
        fs = np.datetime64(fe['start'])
        fe_end = np.datetime64(fe['end'])
        mask = (we_arr >= fs) & (we_arr <= fe_end)
        labels[mask] = 1
    
    return labels


def compute_asset_lookback(window_ends, fail_events, maint_events):
    """Compute days_since_last_failure and days_since_last_revision."""
    fail_ends = sorted(fail_events['end'].values)
    maint_ends = sorted(maint_events['end'].values)
    we_arr = pd.to_datetime(window_ends).values
    
    dsf = np.full(len(we_arr), np.nan, dtype=np.float64)
    dsr = np.full(len(we_arr), np.nan, dtype=np.float64)
    
    for i, we in enumerate(we_arr):
        idx = bisect.bisect_right(fail_ends, we) - 1
        if idx >= 0:
            dsf[i] = (we - fail_ends[idx]) / np.timedelta64(1, 'D')
        
        idx = bisect.bisect_right(maint_ends, we) - 1
        if idx >= 0:
            dsr[i] = (we - maint_ends[idx]) / np.timedelta64(1, 'D')
    
    return dsf, dsr


def load_events(files):
    """Failure and maintenance events from the flag columns only (memory-safe)."""
    df = pd.concat([pd.read_parquet(f, columns=[TS_COL, FAIL_COL, FAIL_TYPE, MAINT_COL]) for f in files], ignore_index=True)
    df[TS_COL] = pd.to_datetime(df[TS_COL])
    df = df.sort_values(TS_COL).reset_index(drop=True)
    return get_failure_events(df), get_maintenance_events(df)


def label_and_filter(feats, fail_events, name):
    """Shifted labels + Helm's filter rule: keep label == 1 OR (label == 0 AND not in failure)."""
    feats = feats.copy()
    feats['label'] = compute_shifted_labels(feats['window_end'], fail_events, HORIZON_S)
    in_fail = compute_current_failure(feats['window_end'], fail_events)
    keep = (feats['label'] == 1) | (in_fail == 0)
    print(f'{name}: {len(feats):,} windows | positives {int(feats["label"].sum()):,} | '
          f'in-failure windows {int(in_fail.sum()):,} | dropped as ambiguous {int((~keep).sum()):,}')
    return feats[keep].reset_index(drop=True), feats


print('Helper functions defined.')

Helper functions defined.


In [3]:
# ── TRAIN: windowed features (cached) + labels ──
train_files = sorted(glob.glob(TRAIN_GLOB, recursive=True))
keep_cols = [TS_COL, FAIL_COL, FAIL_TYPE, MAINT_COL] + ALL_SENSORS
FEAT_CACHE_TRAIN = os.path.join(OUT_DIR, 'supervised', 'features_train_unlabeled.parquet')
os.makedirs(os.path.join(OUT_DIR, 'supervised'), exist_ok=True)

train_fail_events, train_maint_events = load_events(train_files)
print(f'Train failure events: {len(train_fail_events)} | Maintenance events: {len(train_maint_events)}')

if REBUILD_FEATURES or not os.path.exists(FEAT_CACHE_TRAIN):
    print('Building train features from raw data (needs ~7 GB RAM)...')
    chunks = []
    for f in train_files:
        df = pd.read_parquet(f)
        chunks.append(df[[c for c in keep_cols if c in df.columns]])
        del df
    train_raw = pd.concat(chunks, ignore_index=True)
    del chunks; gc.collect()
    train_raw[TS_COL] = pd.to_datetime(train_raw[TS_COL])
    train_raw.sort_values(TS_COL, inplace=True)
    train_raw.reset_index(drop=True, inplace=True)
    train_clean = train_raw[~train_raw[MAINT_COL].astype(bool)].copy()
    del train_raw; gc.collect()
    feats_train = create_windowed_features(train_clean, WINDOW_S, ANALOG_SENSORS, BINARY_SENSORS)
    del train_clean; gc.collect()
    dsf, dsr = compute_asset_lookback(feats_train['window_end'], train_fail_events, train_maint_events)
    feats_train['days_since_last_failure'] = dsf
    feats_train['days_since_last_revision'] = dsr
    feats_train.to_parquet(FEAT_CACHE_TRAIN, index=False)
else:
    feats_train = pd.read_parquet(FEAT_CACHE_TRAIN)
    feats_train['window_end'] = pd.to_datetime(feats_train['window_end'])
    print(f'Loaded cached train features: {feats_train.shape}')

sup_train, _ = label_and_filter(feats_train, train_fail_events, 'Train')
print(f'Supervised train set: {len(sup_train):,} windows, positives {int(sup_train["label"].sum()):,} '
      f'({100 * sup_train["label"].mean():.2f}%), features {len(sup_train.columns) - 3}')
sup_train.to_parquet(os.path.join(OUT_DIR, 'supervised', 'train.parquet'), index=False)
print('Saved to outputs/supervised/train.parquet')
del feats_train; gc.collect()

Train failure events: 10 | Maintenance events: 10
Loaded cached train features: (41375, 342)
Train: 41,375 windows | positives 533 | in-failure windows 308 | dropped as ambiguous 0
Supervised train set: 41,375 windows, positives 533 (1.29%), features 340


Saved to outputs/supervised/train.parquet


0

In [4]:
# ── TEST: windowed features (cached) + labels ──
test_files = sorted(glob.glob(TEST_GLOB, recursive=True))
FEAT_CACHE_TEST = os.path.join(OUT_DIR, 'supervised', 'features_test_unlabeled.parquet')

test_fail_events, test_maint_events = load_events(test_files)
print(f'Test failure events: {len(test_fail_events)} | Maintenance events: {len(test_maint_events)}')
all_fail_events = pd.concat([train_fail_events, test_fail_events], ignore_index=True)
all_maint_events = pd.concat([train_maint_events, test_maint_events], ignore_index=True)

if REBUILD_FEATURES or not os.path.exists(FEAT_CACHE_TEST):
    print('Building test features from raw data...')
    chunks = []
    for f in test_files:
        df = pd.read_parquet(f)
        chunks.append(df[[c for c in keep_cols if c in df.columns]])
        del df
    test_raw = pd.concat(chunks, ignore_index=True)
    del chunks; gc.collect()
    test_raw[TS_COL] = pd.to_datetime(test_raw[TS_COL])
    test_raw.sort_values(TS_COL, inplace=True)
    test_raw.reset_index(drop=True, inplace=True)
    test_clean = test_raw[~test_raw[MAINT_COL].astype(bool)].copy()
    del test_raw; gc.collect()
    feats_test = create_windowed_features(test_clean, WINDOW_S, ANALOG_SENSORS, BINARY_SENSORS)
    del test_clean; gc.collect()
    # Asset lookback uses train + test events (only events that ended before each window)
    dsf, dsr = compute_asset_lookback(feats_test['window_end'], all_fail_events, all_maint_events)
    feats_test['days_since_last_failure'] = dsf
    feats_test['days_since_last_revision'] = dsr
    feats_test.to_parquet(FEAT_CACHE_TEST, index=False)
else:
    feats_test = pd.read_parquet(FEAT_CACHE_TEST)
    feats_test['window_end'] = pd.to_datetime(feats_test['window_end'])
    print(f'Loaded cached test features: {feats_test.shape}')

sup_test, _ = label_and_filter(feats_test, test_fail_events, 'Test')
print(f'Supervised test set: {len(sup_test):,} windows, positives {int(sup_test["label"].sum()):,} '
      f'({100 * sup_test["label"].mean():.2f}%)')
sup_test.to_parquet(os.path.join(OUT_DIR, 'supervised', 'test.parquet'), index=False)
print('Saved outputs/supervised/test.parquet')
del feats_test; gc.collect()

Test failure events: 11 | Maintenance events: 8
Loaded cached test features: (23147, 342)
Test: 23,147 windows | positives 669 | in-failure windows 235 | dropped as ambiguous 0
Supervised test set: 23,147 windows, positives 669 (2.89%)


Saved outputs/supervised/test.parquet


0

In [5]:
# ── Verify supervised datasets ──
sup_train = pd.read_parquet(os.path.join(OUT_DIR, 'supervised', 'train.parquet'))
sup_test  = pd.read_parquet(os.path.join(OUT_DIR, 'supervised', 'test.parquet'))

print('=== Supervised Dataset Summary ===')
print(f'Train: {sup_train.shape} | Test: {sup_test.shape}')
print(f'\nTrain labels: {sup_train["label"].value_counts().to_dict()}')
print(f'Test labels:  {sup_test["label"].value_counts().to_dict()}')
print(f'\nTrain date range: {sup_train["timestamp"].min()} to {sup_train["timestamp"].max()}')
print(f'Test date range:  {sup_test["timestamp"].min()} to {sup_test["timestamp"].max()}')
print(f'\nFeature columns: {len(sup_train.columns) - 3}')
print(f'Columns: {sup_train.columns.tolist()[:10]}...')
print(f'\nMissing values in train: {sup_train.isnull().sum().sum()}')
print(f'Missing values in test:  {sup_test.isnull().sum().sum()}')

=== Supervised Dataset Summary ===
Train: (41375, 343) | Test: (23147, 343)

Train labels: {0: 40842, 1: 533}
Test labels:  {0: 22478, 1: 669}

Train date range: 2024-06-01 04:35:50 to 2025-02-11 23:59:10
Test date range:  2025-02-12 01:03:20 to 2025-06-30 23:58:20

Feature columns: 340
Columns: ['timestamp', 'CW1_MAIN_RESERVOIR_PRESSURE_min', 'CW2_MAIN_RESERVOIR_PRESSURE_min', 'CW1_BRAKE_CYLINDER_PRESSURE_BOGIE1_min', 'CW1_BRAKE_CYLINDER_PRESSURE_BOGIE2_min', 'CW2_BRAKE_CYLINDER_PRESSURE_BOGIE1_min', 'CW2_BRAKE_CYLINDER_PRESSURE_BOGIE2_min', 'MW1_BRAKE_CYLINDER_PRESSURE_BOGIE1_min', 'MW1_BRAKE_CYLINDER_PRESSURE_BOGIE2_min', 'MW2_BRAKE_CYLINDER_PRESSURE_BOGIE1_min']...

Missing values in train: 9808
Missing values in test:  0


## 2. Unsupervised Sequence Creation (LSTM-AE input)

Follows Helm's `references/stefan/create_unsupervised_sequence_datasets.py` with Steiner's best LSTM-AE configuration (Wseq = 23 min, Wlabel = 3.1 h):
- **Labels at the sequence end:** `label_old` = in a failure at window_end; `label_shift` = a failure within the next 3.1 h. Test sequences carry `label_shift`.
- **Normalisation:** mean/std per channel from **normal** train seconds only (not in a failure and no failure within 3.1 h).
- **Sequences:** non-overlapping, epoch-aligned 1402 s windows with at least 80% of the seconds observed; missing seconds forward-filled for on/off signals and linearly interpolated for analog signals. Maintenance rows are removed.
- **Splits:** the last 20% (in time) of the train period is a labelled hold-out (`dev_eval`) for choosing the alarm threshold. From the first 80%, only normal sequences (`label_old` = 0 and `label_shift` = 0) are kept and split 80/20 in time into LSTM-AE training and validation sets.
- **Deviation from Helm:** every 4th second is kept (350 of 1402 steps) to fit CPU memory.

Raw files are streamed one day at a time (a window crossing midnight is completed with the next file), so memory stays low.

> **Status (2026-10-27): aligned with Helm's code but not rebuilt yet (planned for Phase 2).** The cached sequences in `outputs/unsupervised/` still follow the earlier rule (sequence labelled by any failure inside it, no `dev_eval` hold-out), and the cell below prints a warning while they are in use. Before running NB02b, set `REBUILD_SEQUENCES = True` and run this notebook (about an hour), then retrain the LSTM-AE (`RETRAIN_AE = True` in NB02b).

In [ ]:
# ── Sequence settings and helpers (Helm's create_unsupervised_sequence_datasets.py) ──
SEQ_WINDOW_S = 1402      # 23 min sequences (Steiner's best LSTM-AE sequence length)
SEQ_HORIZON_S = 11163    # 3.1 h label horizon (Steiner's best LSTM-AE configuration: 23 min / 3.1 h)
UNSUP_CHANNELS = ALL_SENSORS  # same 95 sensors
SUBSAMPLE = 4  # keep every 4th second: 1402 -> 350 steps (CPU memory; Helm keeps all 1402 steps)
seq_len_sub = SEQ_WINDOW_S // SUBSAMPLE
n_channels = len(UNSUP_CHANNELS)
unsup_cols = [TS_COL, FAIL_COL, MAINT_COL] + UNSUP_CHANNELS
BINARY_IDX = np.array([UNSUP_CHANNELS.index(c) for c in BINARY_SENSORS])
ANALOG_IDX = np.array([UNSUP_CHANNELS.index(c) for c in ANALOG_SENSORS])
SEQ_RULE = 'helm_v1'  # stored in metadata.json; cached sequences built under another rule are flagged
SEQ_DIR = os.path.join(OUT_DIR, 'unsupervised')


def sequence_labels(times, fail_events):
    """Helm's labels, taken at the sequence end (window_end):
    label_old = in a failure at window_end; label_shift = a failure overlaps [window_end, window_end + SEQ_HORIZON_S)."""
    return compute_current_failure(times, fail_events), compute_shifted_labels(times, fail_events, SEQ_HORIZON_S)


def iter_clean(files):
    """One day file at a time: sequence channels, maintenance rows removed (Helm's filter_revisions), sorted.
    Streaming keeps memory low (the whole train period at once would need ~10 GB)."""
    for f in files:
        df = pd.read_parquet(f)
        df = df[[c for c in unsup_cols if c in df.columns]]
        df[TS_COL] = pd.to_datetime(df[TS_COL])
        yield df[~df[MAINT_COL].astype(bool)].sort_values(TS_COL).reset_index(drop=True)


def normal_channel_stats(files, fail_events):
    """Mean/std per channel over NORMAL seconds only: not in a failure and no failure within SEQ_HORIZON_S (Helm)."""
    n = np.zeros(n_channels); s1 = np.zeros(n_channels); s2 = np.zeros(n_channels)
    for df in iter_clean(files):
        old, new = sequence_labels(df[TS_COL], fail_events)
        v = df.loc[(old == 0) & (new == 0), UNSUP_CHANNELS].to_numpy(np.float64)
        ok = ~np.isnan(v)
        n += ok.sum(axis=0); v = np.where(ok, v, 0.0)
        s1 += v.sum(axis=0); s2 += (v ** 2).sum(axis=0)
    mu = s1 / np.maximum(n, 1)
    sd = np.sqrt(np.maximum(s2 - n * mu ** 2, 0) / np.maximum(n - 1, 1))
    return {c: {'mean': float(mu[j]), 'std': float(sd[j]) if sd[j] > 0 else 1.0} for j, c in enumerate(UNSUP_CHANNELS)}


def _window_sequences(ts, vals, mu, sd):
    """Sequences of complete epoch-aligned windows in one block (ts in epoch seconds)."""
    win_id = ts // SEQ_WINDOW_S
    bounds = np.flatnonzero(np.diff(win_id)) + 1
    starts, ends = np.r_[0, bounds], np.r_[bounds, len(ts)]
    grid = np.arange(SEQ_WINDOW_S)
    pick = grid[::SUBSAMPLE][:seq_len_sub]
    seqs, times = [], []
    for s, e in zip(starts, ends):
        if e - s < int(SEQ_WINDOW_S * 0.8):  # Helm: at least 80 % of the seconds observed
            continue
        w0 = win_id[s] * SEQ_WINDOW_S
        offs = ts[s:e] - w0
        v = (vals[s:e] - mu) / sd
        full = np.empty((SEQ_WINDOW_S, n_channels))
        ffill = np.clip(np.searchsorted(offs, grid, side='right') - 1, 0, None)
        full[:, BINARY_IDX] = v[ffill][:, BINARY_IDX]  # binary: forward fill
        for j in ANALOG_IDX:                            # analog: linear interpolation
            full[:, j] = np.interp(grid, offs, v[:, j])
        seqs.append(np.nan_to_num(full[pick], nan=0.0).astype(np.float32))
        times.append(w0 + SEQ_WINDOW_S)
    return seqs, times


def build_sequences(files, stats):
    """Non-overlapping epoch-aligned 23-min sequences (Helm's make_sequences), standardised with `stats`,
    every SUBSAMPLE-th second kept. A window crossing midnight is completed with the next day's file.
    Returns sequences (N, seq_len_sub, n_channels) and their window_end times."""
    mu = np.array([stats[c]['mean'] for c in UNSUP_CHANNELS])
    sd = np.array([stats[c]['std'] for c in UNSUP_CHANNELS])
    seqs, times = [], []
    buf_ts, buf_v = np.empty(0, dtype=np.int64), np.empty((0, n_channels))
    for df in iter_clean(files):
        ts = np.r_[buf_ts, df[TS_COL].values.astype('datetime64[s]').astype(np.int64)]
        vals = np.vstack([buf_v, df[UNSUP_CHANNELS].to_numpy(np.float64)])
        if len(ts) == 0:
            continue
        last = ts[-1] // SEQ_WINDOW_S
        done = ts // SEQ_WINDOW_S < last  # the last window may continue in the next file
        s, t = _window_sequences(ts[done], vals[done], mu, sd)
        seqs += s; times += t
        buf_ts, buf_v = ts[~done], vals[~done]
    if len(buf_ts):
        s, t = _window_sequences(buf_ts, buf_v, mu, sd)
        seqs += s; times += t
    arr = np.stack(seqs) if seqs else np.zeros((0, seq_len_sub, n_channels), dtype=np.float32)
    return arr, pd.to_datetime(np.array(times, dtype='int64'), unit='s')


print(f'Sequences: {SEQ_WINDOW_S} s ({SEQ_WINDOW_S // 60} min), every {SUBSAMPLE}th second -> {seq_len_sub} steps x {n_channels} channels; '
      f'label horizon {SEQ_HORIZON_S} s ({SEQ_HORIZON_S / 3600:.1f} h)')

In [ ]:
# ── TRAIN sequences (cached): stats, sequences, dev_eval hold-out, normal-only train/val (Helm) ──
TRAIN_SEQ_FILES = [os.path.join(SEQ_DIR, f) for f in ('train.npy', 'val.npy', 'metadata.json')]
os.makedirs(SEQ_DIR, exist_ok=True)

if REBUILD_SEQUENCES or not all(os.path.exists(p) for p in TRAIN_SEQ_FILES):
    train_files = sorted(glob.glob(TRAIN_GLOB, recursive=True))
    print('Channel statistics from normal train seconds (not in failure, no failure within the horizon)...')
    channel_stats = normal_channel_stats(train_files, train_fail_events)

    print('Building train sequences (streamed per day file)...')
    dev_seqs, dev_times = build_sequences(train_files, channel_stats)
    label_old, label_shift = sequence_labels(dev_times, train_fail_events)
    print(f'Train-period sequences: {dev_seqs.shape} | in failure at the end: {int(label_old.sum())} | '
          f'failure within {SEQ_HORIZON_S / 3600:.1f} h: {int(label_shift.sum())}')

    # Helm: the last 20 % (in time) of the train period is a labelled hold-out for the alarm threshold (dev_eval);
    # the first 80 % is the training pool, of which only normal sequences (old == 0 and shift == 0) are used.
    eval_cut = dev_times[min(int(len(dev_times) * 0.8), len(dev_times) - 1)]
    in_eval = np.asarray(dev_times >= eval_cut)
    normal = (~in_eval) & (label_old == 0) & (label_shift == 0)
    normal_idx = np.flatnonzero(normal)
    n_tr = int(len(normal_idx) * 0.8)  # temporal 80/20 split of the normal sequences into train / val
    np.save(os.path.join(SEQ_DIR, 'train.npy'), dev_seqs[normal_idx[:n_tr]])
    np.save(os.path.join(SEQ_DIR, 'val.npy'), dev_seqs[normal_idx[n_tr:]])
    np.save(os.path.join(SEQ_DIR, 'dev_eval.npy'), dev_seqs[in_eval])
    np.save(os.path.join(SEQ_DIR, 'dev_eval_labels.npy'), label_shift[in_eval].astype(np.int8))
    np.save(os.path.join(SEQ_DIR, 'dev_eval_times.npy'), np.asarray(dev_times[in_eval]))
    seq_meta = {'rule': SEQ_RULE, 'channels': UNSUP_CHANNELS, 'seq_len_s': SEQ_WINDOW_S, 'subsample': SUBSAMPLE,
                'effective_seq_len': seq_len_sub, 'label_horizon_s': SEQ_HORIZON_S, 'channel_stats': channel_stats,
                'dev_eval_start': str(eval_cut),
                'counts': {'train_period': int(len(dev_times)), 'train': n_tr, 'val': int(len(normal_idx) - n_tr),
                           'dev_eval': int(in_eval.sum()), 'dev_eval_positive': int(label_shift[in_eval].sum())}}
    with open(os.path.join(SEQ_DIR, 'metadata.json'), 'w') as f:
        json.dump(seq_meta, f, indent=2)
    print(f'Saved: {seq_meta["counts"]} (dev_eval from {eval_cut})')
    del dev_seqs; gc.collect()
else:
    with open(os.path.join(SEQ_DIR, 'metadata.json')) as f:
        seq_meta = json.load(f)
    channel_stats = seq_meta['channel_stats']
    print('Using cached train sequences:', np.load(TRAIN_SEQ_FILES[0], mmap_mode='r').shape,
          '| val:', np.load(TRAIN_SEQ_FILES[1], mmap_mode='r').shape)

if seq_meta.get('rule') != SEQ_RULE:
    print(f'WARNING: the cached sequences were built with an older rule ({seq_meta.get("rule", "pre-Helm alignment")}), '
          f'not {SEQ_RULE}. Set REBUILD_SEQUENCES = True before running NB02b (Phase 2).')

In [ ]:
# ── TEST sequences (cached): standardised with the train statistics, labelled like Helm's test set ──
TEST_SEQ_FILES = [os.path.join(SEQ_DIR, f) for f in ('test.npy', 'test_labels.npy', 'test_times.npy')]
if REBUILD_SEQUENCES or not all(os.path.exists(p) for p in TEST_SEQ_FILES):
    test_files = sorted(glob.glob(TEST_GLOB, recursive=True))
    print('Building test sequences (streamed per day file)...')
    test_seqs, test_times = build_sequences(test_files, channel_stats)
    _, test_shift = sequence_labels(test_times, test_fail_events)
    np.save(os.path.join(SEQ_DIR, 'test.npy'), test_seqs)
    np.save(os.path.join(SEQ_DIR, 'test_labels.npy'), test_shift.astype(np.int8))  # Helm: label = label_shift
    np.save(os.path.join(SEQ_DIR, 'test_times.npy'), np.asarray(test_times))
    print(f'Test sequences: {test_seqs.shape} | failure within {SEQ_HORIZON_S / 3600:.1f} h of the end: {int(test_shift.sum())}')
    del test_seqs; gc.collect()
else:
    print('Using cached test sequences:', np.load(TEST_SEQ_FILES[0], mmap_mode='r').shape)

## Traceability: Pipeline Position

**NB01 → NB02a → NB02b → NB03 → NB04 → NB05**

### What this notebook provides:
- Windowed feature datasets: 340 features in 6-min (350 s) windows
- Train: 41,375 windows | Test: 23,147 windows
- Label (Helm's code): failure in [window_end, window_end + 6.2 h); in-failure windows are positive
- LSTM-AE sequences: 23-min sequences of the 95 sensor channels (every 4th second → 350 steps), labelled and split as in Helm's code (3.1 h horizon, `dev_eval` hold-out); rebuild pending for Phase 2

### Outputs:
- outputs/supervised/train.parquet, test.parquet → NB02b, NB03, NB04, NB05
- outputs/unsupervised/train.npy, val.npy, dev_eval*.npy, test.npy, test_labels.npy, test_times.npy, metadata.json → NB02b, NB05